In [1]:
import numpy as np
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import os

# Custom Dataset for MRI images
class CustomImageDataset(Dataset):
    def __init__(self, image_dir, transform=None):
        self.image_dir = image_dir
        self.image_files = [
            f for f in sorted(os.listdir(image_dir))
            if os.path.isfile(os.path.join(image_dir, f)) and f.lower().endswith(('.png', '.jpg', '.jpeg'))
        ]
        self.transform = transform

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_path = os.path.join(self.image_dir, self.image_files[idx])
        img = Image.open(img_path).convert("L")  # Use "RGB" if your MRI images are color
        if self.transform:
            img = self.transform(img)
        return img

# Adjust the path to your MRI images folder
transform = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5]),
])

dataset = CustomImageDataset("ULTRA IMAGES", transform=transform)
dataloader = DataLoader(dataset, batch_size=10, shuffle=True)


In [2]:
from diffusers import UNet2DModel

model = UNet2DModel(
    sample_size=128,
    in_channels=1,   # 1 for grayscale MRI, 3 for RGB MRI
    out_channels=1,  # 1 for grayscale MRI, 3 for RGB MRI
    layers_per_block=3,
    block_out_channels=(128,128,256,256,512,512),
    down_block_types=(
        "DownBlock2D", "DownBlock2D", "DownBlock2D", "DownBlock2D", "AttnDownBlock2D", "DownBlock2D",
    ),
    up_block_types=(
        "UpBlock2D", "AttnUpBlock2D", "UpBlock2D", "UpBlock2D", "UpBlock2D", "UpBlock2D",
    ),
)

2025-05-08 13:06:37.156497: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-05-08 13:06:37.897111: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [3]:
from diffusers import DDPMScheduler

scheduler = DDPMScheduler(num_train_timesteps=2000)


In [5]:
import torch
from torch.optim import Adam

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
optimizer = Adam(model.parameters(), lr=1e-4)

num_epochs = 1000
for epoch in range(num_epochs):
    for batch in dataloader:
        batch = batch.to(device)
        timesteps = torch.randint(0, scheduler.config.num_train_timesteps, (batch.size(0),), device=device).long()
        noise = torch.randn_like(batch)
        noisy_images = scheduler.add_noise(batch, noise, timesteps)
        noise_pred = model(noisy_images, timesteps).sample
        loss = torch.nn.functional.mse_loss(noise_pred, noise)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        torch.cuda.empty_cache()
    print(f"Epoch {epoch+1} completed. Loss: {loss.item()}")


Epoch 1 completed. Loss: 0.06610491871833801
Epoch 2 completed. Loss: 0.03267162665724754
Epoch 3 completed. Loss: 0.015384628437459469
Epoch 4 completed. Loss: 0.011920979246497154
Epoch 5 completed. Loss: 0.013063347898423672
Epoch 6 completed. Loss: 0.011047929525375366
Epoch 7 completed. Loss: 0.016971219331026077
Epoch 8 completed. Loss: 0.018395420163869858
Epoch 9 completed. Loss: 0.005958801135420799
Epoch 10 completed. Loss: 0.008855262771248817
Epoch 11 completed. Loss: 0.017763754352927208
Epoch 12 completed. Loss: 0.023965081200003624
Epoch 13 completed. Loss: 0.006422262638807297
Epoch 14 completed. Loss: 0.0036963517777621746
Epoch 15 completed. Loss: 0.0024120393209159374
Epoch 16 completed. Loss: 0.0037981881760060787
Epoch 17 completed. Loss: 0.03630341961979866
Epoch 18 completed. Loss: 0.003414379432797432
Epoch 19 completed. Loss: 0.0022065332159399986
Epoch 20 completed. Loss: 0.0015499226283282042
Epoch 21 completed. Loss: 0.04905777424573898
Epoch 22 completed. L

In [6]:
model.save_pretrained("Pre-built Diffusion (epoch=1000) ULTRA")

In [ ]:
import torch
from diffusers import DDIMPipeline, DDIMScheduler, UNet2DModel
import matplotlib.pyplot as plt
from safetensors.torch import load_file

device = "cuda" if torch.cuda.is_available() else "cpu"

# Load the trained model
model = UNet2DModel(
    sample_size=128,
    in_channels=1,
    out_channels=1,
    layers_per_block=3,
    block_out_channels=(128,128,256,256,512,512),
    down_block_types=(
        "DownBlock2D", "DownBlock2D", "DownBlock2D", "DownBlock2D", "AttnDownBlock2D", "DownBlock2D",
    ),
    up_block_types=(
        "UpBlock2D", "AttnUpBlock2D", "UpBlock2D", "UpBlock2D", "UpBlock2D", "UpBlock2D",
    ),
).to(device)

state_dict = load_file("Pre-built Diffusion (epoch=5000) ULTRA/diffusion_pytorch_model.safetensors", device=device)
model.load_state_dict(state_dict)
model.eval()

# Setup DDIM pipeline
scheduler = DDIMScheduler(num_train_timesteps=2000)
pipeline = DDIMPipeline(unet=model, scheduler=scheduler).to(device)

# Generate synthesized images
num_samples = 2
num_inference_steps = 20

with torch.no_grad():
    generated_images = pipeline(
        num_inference_steps=num_inference_steps,
        batch_size=num_samples
    ).images

# Plot the results
plt.figure(figsize=(15, 3))
for i, img in enumerate(generated_images):
    plt.subplot(1, num_samples, i + 1)
    plt.imshow(img, cmap="gray")
    plt.axis("off")
    plt.title(f"Sample {i+1}")
plt.show()


In [212]:
from torchvision import transforms
from skimage.metrics import structural_similarity as ssim
import numpy as np

# 1. Convert generated images (PIL) to tensors in [0,1]
to_tensor = transforms.ToTensor()
gen_batch = torch.stack([to_tensor(img) for img in generated_images])  # [10, 1, 128, 128]

# 2. Denormalize real images from [-1,1] to [0,1]
def denorm(img):
    return ((img * 0.5) + 0.5).clamp(0, 1)
# Load one batch of real images from your dataloader
real_batch = next(iter(dataloader))  # shape: [batch_size, 1, 128, 128]
real_batch_01 = denorm(real_batch)

# 3. Compute SSIM
ssim_scores = []
for real, fake in zip(real_batch_01, gen_batch):
    real_np = real.squeeze().cpu().numpy()
    fake_np = fake.squeeze().cpu().numpy()
    score = ssim(real_np, fake_np, data_range=1.0)
    ssim_scores.append(score)

print(f"Mean SSIM: {np.mean(ssim_scores):.4f}")


Mean SSIM: 0.7664
